# Titelgenerator voor HBO Kennisbank

Deze notebook fine-tunet een klein seq2seq-model op hbo_kennisbank_clean.csv om van een abstract een compacte, informatieve titel te genereren.

De opzet is bewust licht gehouden zodat de training past binnen ongeveer 12 GB RAM en binnen een redelijke looptijd.


## 1. Importen en Configuratie

Deze sectie laadt alle benodigde bibliotheken en stelt de hyperparameters in.



**Belangrijkste instellingen:**

- **model_name**: Gebruikt `flan-t5-small` — een licht maar krachtig model voor summarisatie/generatie-taken

- **max_source_length**: 256 tokens voor het abstracten input

- **max_target_length**: 32 tokens voor de gegenereerde titel (compact)

- **max_train_examples**: 400 trainingsvoorbeelden (past in ~12GB RAM)

- **prefix**: Instructie voor het model ("Genereer een korte en informatieve titel: ")



De `set_seed()` zorgt voor reproduceerbare resultaten.


In [2]:
#Imports
from pathlib import Path
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)
import evaluate
from IPython.display import display
from groq import Groq
import re
import time

c:\Users\verav\OneDrive\Documenten\GenAI-Portfolio2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Paden en hyperparameters
csv_path = Path(r"hbo_kennisbank_clean.csv")
model_name = "google/flan-t5-small"
output_dir = Path("title_generator_flan_t5_small")
random_state = 42
max_source_length = 256
max_target_length = 32
max_train_examples = 400
max_val_examples = 100

prefix = "Genereer een korte en informatieve titel: "

set_seed(random_state)
output_dir.mkdir(exist_ok=True)


## 2. Data Laden en Inspecteren

Deze sectie leest het CSV-bestand en zuivert het:

- **Validatie**: Controleert of kolommen "title" en "abstract" aanwezig zijn

- **Schoonmaak**: Verwijdert lege waarden en dubbele rijen

- **Statistieken**: Toont het aantal bruikbare trainingsrijen en voorbeelden



Het uiteindelijke dataset bevat `title` (doel) en `abstract` (invoer).


In [4]:
# 2. Data laden en inspecteren
df = pd.read_csv(csv_path)
expected_columns = {"title", "abstract"}
missing_columns = expected_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Ontbrekende kolommen in CSV: {sorted(missing_columns)}")

df = df.loc[:, ["abstract", "title"]].dropna().copy()
df["abstract"] = df["abstract"].astype(str).str.strip()
df["title"] = df["title"].astype(str).str.strip()
df = df[(df["abstract"] != "") & (df["title"] != "")]
df = df.drop_duplicates(subset=["abstract", "title"]).reset_index(drop=True)

print(f"Aantal bruikbare rijen: {len(df)}")

display(df.head())


Aantal bruikbare rijen: 549


,abstract,title
0,Achtergrond: Het herstel na conservatief behan...,Fysiotherapeutische analyse en conservatieve b...
1,In deze scriptie zijn 144 metalen militaria ui...,Civiel centrum Oppidum Batavorum
2,The escalating plastic waste crisis has increa...,Comparative analysis of fingermark development...
3,Artikel over de rol van passie in de amateurkunst,Passie in de amateurkunst
4,Dit rapport is deel I van twee delen waarin de...,A legal overview on International & European S...


## 3. Voorbewerking, Split en Tokenisatie

Deze sectie bereidt de data voor voor training:

1. **Train/Validatie Split**: 80/20 verdeling, max 400 train en 100 validatie rijen

2. **Hugging Face Datasets**: Converteert DataFrames naar `Dataset` objecten

3. **Tokenisatie**:

   - Voegt prefix toe aan abstracten (instructie-engineering)

   - Codeert input en labels tot token-IDs

   - Trunceert lange teksten naar max_source_length / max_target_length

   - Output: `tokenized` bevat `input_ids`, `attention_mask` en `labels`


In [5]:
# 3. Voorbewerking, split en tokenisatie
train_df, val_df = train_test_split(df, test_size=0.2, random_state=random_state)
train_df = train_df.head(max_train_examples).reset_index(drop=True)
val_df = val_df.head(max_val_examples).reset_index(drop=True)

dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "validation": Dataset.from_pandas(val_df, preserve_index=False),
    }
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_batch(batch):
    inputs = [prefix + text for text in batch["abstract"]]
    model_inputs = tokenizer(inputs, max_length=max_source_length, truncation=True)
    labels = tokenizer(text_target=batch["title"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

display(train_df.head(3))

display(val_df.head(3))


Map: 100%|██████████| 100/100 [00:00<00:00, 1496.20 examples/s]


,abstract,title
0,Dysfagie is een symptoom dat wordt gekenmerkt ...,De TotalCup® in de zorgpraktijk: een kwalitati...
1,In deze lijst staan alle interventie-ideeën be...,Lijst met interventie-ideeën van mbo-studenten
2,Met toestemming vqn de uitgever overgenomen ui...,Soenveld_dewit_19122025_essayrobotrechten_nede...


,abstract,title
0,Het objectief meten van spierkracht is essenti...,De intra-rater betrouwbaarheid van de citecmot...
1,Qualitative research in the form of interviews...,Improving the feedback systems at Van der Lind...
2,El vidrio aislante viejo suele acabar en aplic...,Vidrio Plano 100% Reutilizado: Dando una Segun...


## 4. Model Fine-tunen en Evalueren

Deze sectie configureert het fine-tuning proces:



**Evaluatiemetrieken:**

- **ROUGE** (Recall-Oriented Understudy for Gisting Evaluation): Meet lexicale overlap tussen gegenereerde en referentie-titels

- `compute_metrics()` decodeert voorspellingen en labels terug naar tekst, dan berekent ROUGE-scores



**Training Parameters:**

- **Learning rate**: 3e-4 (laag, geschikt voor fine-tuning van voorgetrainde modellen)

- **Batch size**: 4 per device, gradient accumulation=2 (equivalent ~16)

- **Epochs**: 2 (voorzichtig gekozen voor kleine dataset)

- **Beam search**: 4 beams voor generatie (betere kwaliteit dan greedy)

- **Early stopping**: Gebaseerd op rougeL metric



**Opmerking**: De `trainer.train()` is uitgecomment om onbedoelde training te voorkomen.


In [6]:
# 4. Model fine-tunen en evalueren
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
rouge = evaluate.load("rouge")


def compute_metrics(eval_preds):
    import numpy as np

    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Convert tensors to numpy
    try:
        if hasattr(predictions, "cpu"):
            predictions = predictions.cpu().numpy()
    except Exception:
        pass
    try:
        if hasattr(labels, "cpu"):
            labels = labels.cpu().numpy()
    except Exception:
        pass

    # Replace negative values (including -100) with pad token id for safe decoding
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    if predictions is not None:
        predictions = np.where(predictions < 0, pad_id, predictions)
    if labels is not None:
        labels = np.where(labels < 0, pad_id, labels)

    # Ensure integer dtype for tokenizer
    predictions = predictions.astype(np.int64)
    if labels is not None:
        labels = labels.astype(np.int64)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = (
        tokenizer.batch_decode(labels, skip_special_tokens=True) if labels is not None else [""] * len(decoded_preds)
    )

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [lab.strip() for lab in decoded_labels]

    scores = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {metric: round(value, 4) for metric, value in scores.items()}


training_args = Seq2SeqTrainingArguments(
    output_dir=str(output_dir),
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=max_target_length,
    generation_num_beams=4,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2119.24it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


# Training


In [7]:
# Veiligheids-flag om training expliciet te starten
run_training = True

# Training
if run_training:
    trainer.train()
    eval_results = trainer.evaluate()
    print(eval_results)
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
else:
    print("Training is uitgeschakeld. Zet `run_training = True` in de configuratiecel om te starten.")


c:\Users\verav\OneDrive\Documenten\GenAI-Portfolio2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,6.847442,3.248805,0.200300,0.076000,0.168600,0.170400
2,5.966657,3.271343,0.207300,0.089200,0.179600,0.180900


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.45it/s]
c:\Users\verav\OneDrive\Documenten\GenAI-Portfolio2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.11it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].
c:\Users\verav\OneDrive\Documenten\GenAI-Portfolio2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum
5.966657,3.271343,2,0.207300,0.089200,0.179600,0.180900


{'eval_loss': 3.2713425159454346, 'eval_rouge1': 0.2073, 'eval_rouge2': 0.0892, 'eval_rougeL': 0.1796, 'eval_rougeLsum': 0.1809}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.35it/s]


## 5. Generatie, Sampling en Strategie-Evaluatie

Deze sectie vergelijkt drie generatie-strategieën voor titel-generatie:



**Sampling Strategieën:**

1. **Greedy Decoding** (`do_sample=False, num_beams=1`)

   - Kiest altijd het meest waarschijnlijke token

   - Voordeel: Deterministisch, snel, consistent

   - Nadeel: Kan monotoon of repetitief zijn



2. **Beam Search** (`num_beams=4`)

   - Volgt meerdere paden parallel, selecteert beste volgorde

   - Voordeel: Beter vocabulaire en zinsbouw dan greedy

   - Nadeel: Langzamer, soms nog deterministisch



3. **Top-p (Nucleus) Sampling** (`do_sample=True, top_p=0.9, temperature=0.8`)

   - Samplet uit de waarschijnlijkheidsverdeling, beperkt tot top 90% cumulatieve kans

   - Temperature=0.8: Minder random dan standaard (1.0)

   - Voordeel: Meer variatie, natuurlijker klinkend

   - Nadeel: Non-deterministisch



**Aanbevolen keuze**: **Beam Search** voor productie (consistent + kwaliteit) of **Top-p** voor meer variatie.


In [8]:
# 5. Generatie, sampling en evaluatie
model = trainer.model
model.eval()
device = model.device

def generate_title(abstract, **generation_kwargs):
    prompt = prefix + abstract
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_source_length).to(device)
    generated = model.generate(
        **inputs,
        #max_new_tokens=max_target_length,
        **generation_kwargs,
    )

    return tokenizer.decode(generated[0], skip_special_tokens=True).strip()



generation_strategies = {
    "greedy": {"do_sample": False, "num_beams": 1},
    "beam_4": {"do_sample": False, "num_beams": 4},
    "top_p": {"do_sample": True, "top_p": 0.9, "temperature": 0.8, "num_beams": 1},
}

sample_rows = val_df.head(5).copy()

for strategy_name, kwargs in generation_strategies.items():
    print(f"\n=== {strategy_name} ===")
    for _, row in sample_rows.iterrows():
        prediction = generate_title(row["abstract"], **kwargs)
        print("REF :", row["title"])
        print("PRED:", prediction)
        print("---")



def evaluate_strategy(strategy_kwargs, sample_size=25):
    subset = val_df.head(sample_size)
    predictions = [generate_title(abstract, **strategy_kwargs) for abstract in subset["abstract"]]
    references = subset["title"].tolist()

    return rouge.compute(predictions=predictions, references=references, use_stemmer=True)



strategy_scores = {
    name: evaluate_strategy(kwargs, sample_size=25)
    for name, kwargs in generation_strategies.items()
}

strategy_scores



=== greedy ===
REF : De intra-rater betrouwbaarheid van de citecmotion hand-held dynamometer bij het meten van knijpkracht en knie-extensiekracht bij gezonde volwassenen
PRED: CitecMotion Hand Held Dynamometer
---
REF : Improving the feedback systems at Van der Linde Catering.
PRED: Interviews in Van der Linde Catering
---
REF : Vidrio Plano 100% Reutilizado: Dando una Segunda Vida a Cada Panel de Vidrio
PRED: Aislante en el vidrio aislante
---
REF : Ondergrondse energieopslag in Twente Governance arrangementen
PRED: Ondergrondse opslag
---
REF : Wat leren leerlingen en studenten over democratie? Een analyse van lesmethoden maatschappijleer en mbo-burgerschap
PRED: De democratische erosie het versterken van een
---

=== beam_4 ===
REF : De intra-rater betrouwbaarheid van de citecmotion hand-held dynamometer bij het meten van knijpkracht en knie-extensiekracht bij gezonde volwassenen
PRED: CitecMotion Hand Held Dynamometer
---
REF : Improving the feedback systems at Van der Linde Cater

{'greedy': {'rouge1': np.float64(0.3038980403109436),
  'rouge2': np.float64(0.13334499971564595),
  'rougeL': np.float64(0.25780587751555495),
  'rougeLsum': np.float64(0.2577504569504569)},
 'beam_4': {'rouge1': np.float64(0.3349294640716167),
  'rouge2': np.float64(0.1803934165673296),
  'rougeL': np.float64(0.30364964000070527),
  'rougeLsum': np.float64(0.30134807055784724)},
 'top_p': {'rouge1': np.float64(0.20119201957462826),
  'rouge2': np.float64(0.09327812284334022),
  'rougeL': np.float64(0.18053020006421538),
  'rougeLsum': np.float64(0.17819397703745532)}}

## 6. RLAIF (Reinforcement Learning from AI Feedback) en Vervolgstappen



**RLAIF Concept:**

In plaats van handmatige feedback van mensen, gebruiken we een AI (bijv. GPT-4 of een expert LLM) om gegenereerde titels te beoordelen. Deze feedback trainen we vervolgens terug in het model.



**Stappen voor RLAIF:**

1. Genereer kandidaat-titels met het fine-tuned model

2. Score deze titels met een evaluator-LLM op bijv. relevantie, conciseness, informatieviteit (schaal 0-10)

3. Verzamel deze scores in een `rlaif_feedback.csv` met kolommen: `abstract`, `candidate_title`, `score`

4. Voer optioneel een extra training-stap uit waarbij scores als reward gebruikt worden




In [9]:
#client aanmaken
with open("api_key.txt", "r") as f:
    API_KEY = f.read().strip()

client = Groq(api_key=API_KEY)

In [10]:
# Functie voor AI feedback
def score_title_with_ai(abstract, title):

    prompt = f"""
    Score deze academische titel van 0-10.

    Criteria:
    - relevantie
    - duidelijkheid
    - academische stijl

    Geef alleen een getal.

    Abstract:
    {abstract}

    Titel:
    {title}
    """

    #versturen van prompt naar evaluator-model
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    #output van model ophalen
    text = response.choices[0].message.content

    print(text)
    
    #numerieke score uit tekst halen
    match = re.search(r"\d+(\.\d+)?", text)

    if match:
        return float(match.group())

    return None


In [11]:
#kleine subset selecteren
subset = val_df.head(5)

rlaif_rows = []

In [12]:
#RLAIF evaluatie uitvoeren
for _, row in subset.iterrows():

    abstract = row["abstract"]
    real_title = row["title"]

    #verschillende generation strategieën testen
    for strategy_name, kwargs in generation_strategies.items():

        # Titel genereren
        candidate_title = generate_title(
            abstract,
            **kwargs
        )

        print("\n===================================")
        print("STRATEGY:", strategy_name)
        print("REAL TITLE:", real_title)
        print("GENERATED TITLE:", candidate_title)

        # AI score ophalen
        try:

            score = score_title_with_ai(
                abstract,
                candidate_title
            )

        except Exception as e:

            print("ERROR:", e)

            score = None

        print("AI SCORE:", score)

        # Resultaten Opslaan
        rlaif_rows.append({
            "abstract": abstract,
            "real_title": real_title,
            "candidate_title": candidate_title,
            "strategy": strategy_name,
            "score": score
        })

        # Kleine delay tegen rate limits
        time.sleep(2)


STRATEGY: greedy
REAL TITLE: De intra-rater betrouwbaarheid van de citecmotion hand-held dynamometer bij het meten van knijpkracht en knie-extensiekracht bij gezonde volwassenen
GENERATED TITLE: CitecMotion Hand Held Dynamometer
6
AI SCORE: 6.0

STRATEGY: beam_4
REAL TITLE: De intra-rater betrouwbaarheid van de citecmotion hand-held dynamometer bij het meten van knijpkracht en knie-extensiekracht bij gezonde volwassenen
GENERATED TITLE: CitecMotion Hand Held Dynamometer
5
AI SCORE: 5.0

STRATEGY: top_p
REAL TITLE: De intra-rater betrouwbaarheid van de citecmotion hand-held dynamometer bij het meten van knijpkracht en knie-extensiekracht bij gezonde volwassenen
GENERATED TITLE: CitecMotion Hand Held Dynamometer
7
AI SCORE: 7.0

STRATEGY: greedy
REAL TITLE: Improving the feedback systems at Van der Linde Catering.
GENERATED TITLE: Interviews in Van der Linde Catering
6
AI SCORE: 6.0

STRATEGY: beam_4
REAL TITLE: Improving the feedback systems at Van der Linde Catering.
GENERATED TITLE: 

In [13]:
rlaif_df = pd.DataFrame(rlaif_rows)

In [14]:
print("\nRLAIF RESULTATEN:")

display(rlaif_df)


RLAIF RESULTATEN:


,abstract,real_title,candidate_title,strategy,score
0,Het objectief meten van spierkracht is essenti...,De intra-rater betrouwbaarheid van de citecmot...,CitecMotion Hand Held Dynamometer,greedy,6.0
1,Het objectief meten van spierkracht is essenti...,De intra-rater betrouwbaarheid van de citecmot...,CitecMotion Hand Held Dynamometer,beam_4,5.0
2,Het objectief meten van spierkracht is essenti...,De intra-rater betrouwbaarheid van de citecmot...,CitecMotion Hand Held Dynamometer,top_p,7.0
3,Qualitative research in the form of interviews...,Improving the feedback systems at Van der Lind...,Interviews in Van der Linde Catering,greedy,6.0
4,Qualitative research in the form of interviews...,Improving the feedback systems at Van der Lind...,A new feedback system for Van der Linde Catering,beam_4,9.0
5,Qualitative research in the form of interviews...,Improving the feedback systems at Van der Lind...,Interviews and feedback systems at Van der Lin...,top_p,5.0
6,El vidrio aislante viejo suele acabar en aplic...,Vidrio Plano 100% Reutilizado: Dando una Segun...,Aislante en el vidrio aislante,greedy,7.0
7,El vidrio aislante viejo suele acabar en aplic...,Vidrio Plano 100% Reutilizado: Dando una Segun...,La vidrio aislante viejo suele acabar,beam_4,5.0
8,El vidrio aislante viejo suele acabar en aplic...,Vidrio Plano 100% Reutilizado: Dando una Segun...,Aislante aislante del vidrio ais,top_p,5.0
9,Rapport over governance arrangementen waarmee ...,Ondergrondse energieopslag in Twente Governanc...,Ondergrondse opslag,greedy,6.0


In [15]:
# Resultaten opslaan als CSV
rlaif_df.to_csv(
    "rlaif_feedback.csv",
    index=False
)

print("\nCSV opgeslagen als: rlaif_feedback.csv")



CSV opgeslagen als: rlaif_feedback.csv


In [16]:
# Gemiddelde score per strategie berekenen
strategy_scores = (
    rlaif_df.groupby("strategy")["score"]
    .mean()
    .sort_values(ascending=False)
)

print("\nGEMIDDELDE SCORE PER STRATEGIE:")

print(strategy_scores)


GEMIDDELDE SCORE PER STRATEGIE:
strategy
greedy    6.6
top_p     6.6
beam_4    6.0
Name: score, dtype: float64
